<a href="https://colab.research.google.com/github/VasilisPapageorgiou/Cameroon/blob/main/RealCameroonExp1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================
# EXPERIMENT 1
#
# Complete one-cell implementation
#
# Stage A
# Daily cumulative data are converted to weekly incidence.
#
# Stage B
# Deterministic SIR profiling over admissible
# N_eff, S(0), and I(0), subject to R(0)=60.
#
# Stage C
# The deterministic configuration with the smallest
# penalized negative log-likelihood is selected.
#
# Stage D
# Conditional on the selected discrete configuration,
# b, gamma, and phi are re-estimated using the exact
# finite-population CTMC reward-process mean.
#
# Stage E
# Hessian uncertainty, final-size distribution,
# extinction-time moments, and parametric-bootstrap
# max-t simultaneous bands are computed.
# ============================================================

from __future__ import annotations

import time
from dataclasses import dataclass
from pathlib import Path
from typing import Callable

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from IPython.display import display
from scipy.integrate import solve_ivp
from scipy.optimize import minimize
from scipy.special import gammaln
from scipy.sparse import bmat, coo_matrix, csr_matrix
from scipy.sparse.linalg import expm_multiply
from scipy.stats import nbinom


TOTAL_START = time.perf_counter()


# ============================================================
# 1. CONFIGURATION
# ============================================================

PSI = 0.0
NUMBER_OF_REGIMES = 1
R0_FIXED = 60

assert PSI == 0.0
assert NUMBER_OF_REGIMES == 1
assert R0_FIXED >= 0


# Analysis window

ANALYSIS_START = pd.Timestamp("2025-11-16")
ANALYSIS_END = pd.Timestamp("2026-04-24")

WEEK_FREQUENCY = "W-SUN"
EXPECTED_WEEKS = 24


# False reproduces the 24-bin weekly analysis.
#
# The first and last partial calendar weeks are treated as
# full observation intervals in the likelihood.

USE_ACTUAL_PARTIAL_WEEK_LENGTHS = False


# Candidate effective populations

N_GRID = np.array(
    [
        125,
        150,
        175,
        200,
        225,
        250,
    ],
    dtype=int,
)


# Candidate initial infectious populations

I0_GRID = np.arange(
    1,
    11,
    dtype=int,
)


# Recovery-rate penalty

GAMMA_PENALTY_CENTER = np.log(
    1.0 / 3.0
)

GAMMA_PENALTY_SD = 0.18


# Bounds for
#
# eta = (log b, log gamma, log phi)

LOG_PARAMETER_BOUNDS = [
    (
        np.log(0.01),
        np.log(3.00),
    ),
    (
        np.log(0.03),
        np.log(1.50),
    ),
    (
        np.log(0.05),
        np.log(100.0),
    ),
]


# Starting values used during deterministic screening

DETERMINISTIC_PARAMETER_STARTS = [
    np.array(
        [
            0.443,
            0.210,
            4.174,
        ],
        dtype=float,
    ),
    np.array(
        [
            0.700,
            1.0 / 3.0,
            4.000,
        ],
        dtype=float,
    ),
]


# Additional starts used in the exact conditional fit

EXACT_PARAMETER_STARTS = [
    np.array(
        [
            0.350,
            1.0 / 3.0,
            3.000,
        ],
        dtype=float,
    ),
    np.array(
        [
            0.700,
            0.250,
            5.000,
        ],
        dtype=float,
    ),
    np.array(
        [
            1.000,
            0.300,
            2.000,
        ],
        dtype=float,
    ),
]


# Thresholds are constructed after the deterministic
# configuration has been selected.

TAIL_FRACTIONS = np.array(
    [
        0.50,
        0.60,
        0.70,
        0.80,
        0.90,
    ],
    dtype=float,
)


# Bootstrap settings
#
# Use 49 for debugging.
# Use 199 or 499 for manuscript results.

N_BOOTSTRAP_REPLICATES = 199

MAX_BOOTSTRAP_FAILURE_RATE = 0.20
ALPHA = 0.05
BASE_SEED = 20260807


# Output directory

OUTPUT_DIR = Path(
    "/content/experiment1_deterministic_then_exact"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


# ============================================================
# 2. LOCATE DATASET
# ============================================================

def locate_dataset() -> Path:
    candidates = [
        Path("/content/Cameroonmpox.csv"),
        Path("/mnt/data/Cameroonmpox.csv"),
        Path("Cameroonmpox.csv"),
    ]

    for candidate in candidates:
        if candidate.exists():
            return candidate

    try:
        from google.colab import files

        print("Upload Cameroonmpox.csv")

        uploaded = files.upload()

        if not uploaded:
            raise FileNotFoundError(
                "No dataset was uploaded."
            )

        filename = next(iter(uploaded))

        return Path("/content") / filename

    except ImportError as exc:
        raise FileNotFoundError(
            "Cameroonmpox.csv was not found."
        ) from exc


def identify_column(
    data: pd.DataFrame,
    preferred_names: list[str],
    required_tokens: list[str],
) -> str:
    normalized = {
        str(column).strip().lower(): column
        for column in data.columns
    }

    for preferred_name in preferred_names:
        key = preferred_name.strip().lower()

        if key in normalized:
            return str(
                normalized[key]
            )

    for column in data.columns:
        name = str(column).lower()

        if all(
            token in name
            for token in required_tokens
        ):
            return str(column)

    raise ValueError(
        "A required column could not be identified. "
        f"Available columns are {list(data.columns)}."
    )


# ============================================================
# 3. DAILY TO WEEKLY AGGREGATION
# ============================================================

def prepare_weekly_incidence(
    dataset_path: Path,
) -> tuple[
    np.ndarray,
    np.ndarray,
    pd.DataFrame,
    pd.DataFrame,
]:
    raw = pd.read_csv(
        dataset_path,
        sep=None,
        engine="python",
    )

    print("Raw columns")

    print(
        list(raw.columns)
    )

    date_column = identify_column(
        raw,
        preferred_names=[
            "date",
            "observation_date",
            "report_date",
        ],
        required_tokens=[
            "date",
        ],
    )

    cumulative_column = identify_column(
        raw,
        preferred_names=[
            "total_cases",
            "cumulative_cases",
            "cum_cases",
            "case_total",
        ],
        required_tokens=[
            "case",
        ],
    )

    print(
        "Date column",
        date_column,
    )

    print(
        "Cumulative-count column",
        cumulative_column,
    )

    cleaned = pd.DataFrame(
        {
            "date": pd.to_datetime(
                raw[date_column],
                errors="coerce",
            ),
            "total_cases": pd.to_numeric(
                raw[cumulative_column],
                errors="coerce",
            ),
        }
    )

    cleaned = (
        cleaned
        .dropna()
        .sort_values("date")
        .groupby(
            "date",
            as_index=False,
        )["total_cases"]
        .last()
        .reset_index(drop=True)
    )

    if cleaned.empty:
        raise ValueError(
            "No valid observations were found."
        )

    if (
        cleaned["date"].min()
        > ANALYSIS_START
    ):
        raise ValueError(
            "The dataset begins after the analysis start."
        )

    if (
        cleaned["date"].max()
        < ANALYSIS_END
    ):
        raise ValueError(
            "The dataset ends before the analysis end."
        )

    if np.any(
        cleaned["total_cases"].to_numpy()
        < 0
    ):
        raise ValueError(
            "Negative cumulative values were detected."
        )

    full_daily_index = pd.date_range(
        start=cleaned["date"].min(),
        end=cleaned["date"].max(),
        freq="D",
    )

    daily_cumulative = (
        cleaned
        .set_index("date")["total_cases"]
        .reindex(full_daily_index)
        .ffill()
    )

    if daily_cumulative.isna().any():
        raise ValueError(
            "The daily cumulative series contains "
            "unresolved missing values."
        )

    daily_incidence = (
        daily_cumulative.diff()
    )

    daily_incidence.iloc[0] = (
        daily_cumulative.iloc[0]
    )

    negative_locations = np.where(
        daily_incidence.to_numpy()
        < -1e-8
    )[0]

    if len(negative_locations) > 0:
        negative_dates = (
            daily_incidence.index[
                negative_locations
            ]
            .strftime("%Y-%m-%d")
            .tolist()
        )

        raise ValueError(
            "The cumulative series decreases on dates "
            f"{negative_dates}."
        )

    daily_incidence = np.maximum(
        daily_incidence,
        0.0,
    )

    daily_table = pd.DataFrame(
        {
            "date":
                daily_cumulative.index,
            "total_cases":
                daily_cumulative.to_numpy(),
            "daily_cases":
                daily_incidence.to_numpy(),
        }
    )

    analysis_daily = (
        daily_table.loc[
            (
                daily_table["date"]
                >= ANALYSIS_START
            )
            & (
                daily_table["date"]
                <= ANALYSIS_END
            )
        ]
        .copy()
        .set_index("date")
    )

    if analysis_daily.empty:
        raise ValueError(
            "The analysis window contains no observations."
        )

    analysis_daily[
        "week_label"
    ] = (
        analysis_daily.index
        .to_period(
            WEEK_FREQUENCY
        )
        .end_time
        .normalize()
    )

    weekly_table = (
        analysis_daily
        .reset_index()
        .groupby(
            "week_label",
            as_index=False,
        )
        .agg(
            week_start=(
                "date",
                "min",
            ),
            week_end=(
                "date",
                "max",
            ),
            number_of_days=(
                "daily_cases",
                "size",
            ),
            weekly_cases=(
                "daily_cases",
                "sum",
            ),
            cumulative_end=(
                "total_cases",
                "last",
            ),
        )
    )

    weekly_table["week"] = np.arange(
        1,
        len(weekly_table) + 1,
    )

    weekly_table = weekly_table[
        [
            "week",
            "week_label",
            "week_start",
            "week_end",
            "number_of_days",
            "weekly_cases",
            "cumulative_end",
        ]
    ]

    weekly_cases = (
        weekly_table["weekly_cases"]
        .to_numpy(dtype=float)
    )

    if not np.allclose(
        weekly_cases,
        np.round(weekly_cases),
        atol=1e-8,
    ):
        raise ValueError(
            "Weekly incidence is not integer-valued."
        )

    weekly_cases = np.round(
        weekly_cases
    ).astype(int)

    weekly_table["weekly_cases"] = (
        weekly_cases
    )

    if len(weekly_cases) != EXPECTED_WEEKS:
        raise RuntimeError(
            "The aggregation must produce exactly "
            f"{EXPECTED_WEEKS} weekly observations. "
            f"It produced {len(weekly_cases)}."
        )

    if USE_ACTUAL_PARTIAL_WEEK_LENGTHS:
        interval_lengths = (
            weekly_table[
                "number_of_days"
            ]
            .to_numpy(dtype=float)
            / 7.0
        )

    else:
        interval_lengths = np.ones(
            len(weekly_cases),
            dtype=float,
        )

    print()
    print("Weekly data summary")

    print(
        {
            "number_of_weeks":
                len(weekly_cases),
            "total_cases":
                int(
                    weekly_cases.sum()
                ),
            "minimum_weekly_cases":
                int(
                    weekly_cases.min()
                ),
            "maximum_weekly_cases":
                int(
                    weekly_cases.max()
                ),
        }
    )

    print()
    print("Weekly incidence vector")

    print(
        weekly_cases.tolist()
    )

    return (
        weekly_cases,
        interval_lengths,
        weekly_table,
        daily_table,
    )


DATA_PATH = locate_dataset()

(
    Y,
    INTERVAL_LENGTHS,
    WEEKLY_DATA,
    DAILY_DATA,
) = prepare_weekly_incidence(
    DATA_PATH
)

T = len(Y)

OBSERVED_TOTAL = int(
    Y.sum()
)

display(
    WEEKLY_DATA
)

WEEKLY_DATA.to_csv(
    OUTPUT_DIR
    / "weekly_incidence_data.csv",
    index=False,
)

DAILY_DATA.to_csv(
    OUTPUT_DIR
    / "daily_cleaned_data.csv",
    index=False,
)


# ============================================================
# 4. NEGATIVE-BINOMIAL LIKELIHOOD
# ============================================================

def negative_binomial_loglikelihood(
    observations: np.ndarray,
    mean: np.ndarray,
    phi: float,
) -> float:
    observations = np.asarray(
        observations,
        dtype=float,
    )

    mean = np.maximum(
        np.asarray(
            mean,
            dtype=float,
        ),
        1e-12,
    )

    if (
        phi <= 0.0
        or not np.isfinite(phi)
    ):
        return -np.inf

    value = np.sum(
        gammaln(
            observations + phi
        )
        - gammaln(phi)
        - gammaln(
            observations + 1.0
        )
        + phi
        * (
            np.log(phi)
            - np.log(
                phi + mean
            )
        )
        + observations
        * (
            np.log(mean)
            - np.log(
                phi + mean
            )
        )
    )

    return float(
        value
    )


def penalized_nll_from_mean(
    eta: np.ndarray,
    expected_mean: np.ndarray,
    observations: np.ndarray,
) -> float:
    eta = np.asarray(
        eta,
        dtype=float,
    )

    observations = np.asarray(
        observations,
        dtype=float,
    )

    expected_mean = np.asarray(
        expected_mean,
        dtype=float,
    )

    if (
        len(eta) != 3
        or np.any(
            ~np.isfinite(eta)
        )
        or len(expected_mean)
        != len(observations)
    ):
        return 1e100

    phi = float(
        np.exp(
            eta[2]
        )
    )

    loglikelihood = (
        negative_binomial_loglikelihood(
            observations=observations,
            mean=expected_mean,
            phi=phi,
        )
    )

    gamma_penalty = 0.5 * (
        (
            eta[1]
            - GAMMA_PENALTY_CENTER
        )
        / GAMMA_PENALTY_SD
    ) ** 2

    value = (
        -loglikelihood
        + gamma_penalty
    )

    return (
        float(value)
        if np.isfinite(value)
        else 1e100
    )


# ============================================================
# 5. DETERMINISTIC CONFIGURATION CANDIDATES
# ============================================================

def construct_initial_state_candidates(
    population_size: int,
    observed_total: int,
) -> list[
    tuple[
        int,
        int,
        int,
    ]
]:
    candidates = []

    for infectious_initial in I0_GRID:
        susceptible_initial = (
            population_size
            - infectious_initial
            - R0_FIXED
        )

        if susceptible_initial < 1:
            continue

        if susceptible_initial < observed_total:
            continue

        if (
            population_size
            - susceptible_initial
            - infectious_initial
            != R0_FIXED
        ):
            raise RuntimeError(
                "The candidate violates R(0)=60."
            )

        candidates.append(
            (
                population_size,
                susceptible_initial,
                infectious_initial,
            )
        )

    return candidates


ALL_CONFIGURATIONS = []

for candidate_population in N_GRID:
    ALL_CONFIGURATIONS.extend(
        construct_initial_state_candidates(
            population_size=int(
                candidate_population
            ),
            observed_total=OBSERVED_TOTAL,
        )
    )

ALL_CONFIGURATIONS = sorted(
    set(
        ALL_CONFIGURATIONS
    )
)

if not ALL_CONFIGURATIONS:
    raise RuntimeError(
        "No admissible deterministic configurations "
        "were generated."
    )

print()
print(
    "Number of deterministic configurations",
    len(ALL_CONFIGURATIONS),
)


# ============================================================
# 6. DETERMINISTIC SIR MEAN
# ============================================================

def deterministic_expected_incidence(
    population_size: int,
    susceptible_initial: int,
    infectious_initial: int,
    b: float,
    gamma: float,
) -> np.ndarray:
    evaluation_times = np.concatenate(
        [
            np.array(
                [
                    0.0
                ]
            ),
            np.cumsum(
                INTERVAL_LENGTHS
            ),
        ]
    )

    def sir_system(
        time_value: float,
        state: np.ndarray,
    ) -> np.ndarray:
        susceptible = max(
            float(
                state[0]
            ),
            0.0,
        )

        infectious = max(
            float(
                state[1]
            ),
            0.0,
        )

        infection_rate = (
            b
            * susceptible
            * infectious
            / population_size
        )

        return np.array(
            [
                -infection_rate,
                infection_rate
                - gamma
                * infectious,
                infection_rate,
            ],
            dtype=float,
        )

    initial_state = np.array(
        [
            float(
                susceptible_initial
            ),
            float(
                infectious_initial
            ),
            0.0,
        ],
        dtype=float,
    )

    solution = solve_ivp(
        sir_system,
        t_span=(
            0.0,
            float(
                evaluation_times[-1]
            ),
        ),
        y0=initial_state,
        t_eval=evaluation_times,
        method="RK45",
        rtol=2e-6,
        atol=1e-8,
    )

    if not solution.success:
        raise RuntimeError(
            "Deterministic integration failed."
        )

    cumulative_infections = (
        solution.y[2]
    )

    weekly_mean = np.diff(
        cumulative_infections
    )

    return np.maximum(
        weekly_mean,
        1e-12,
    )


# ============================================================
# 7. DETERMINISTIC PARAMETER ESTIMATION
# ============================================================

def fit_deterministic_configuration(
    population_size: int,
    susceptible_initial: int,
    infectious_initial: int,
) -> dict:
    def objective(
        eta: np.ndarray,
    ) -> float:
        eta = np.asarray(
            eta,
            dtype=float,
        )

        b = float(
            np.exp(
                eta[0]
            )
        )

        gamma = float(
            np.exp(
                eta[1]
            )
        )

        try:
            expected_mean = (
                deterministic_expected_incidence(
                    population_size=
                        population_size,
                    susceptible_initial=
                        susceptible_initial,
                    infectious_initial=
                        infectious_initial,
                    b=b,
                    gamma=gamma,
                )
            )

            return penalized_nll_from_mean(
                eta=eta,
                expected_mean=
                    expected_mean,
                observations=Y,
            )

        except Exception:
            return 1e100

    best_fit = None

    for parameter_start in (
        DETERMINISTIC_PARAMETER_STARTS
    ):
        fit = minimize(
            objective,
            x0=np.log(
                parameter_start
            ),
            method="L-BFGS-B",
            bounds=LOG_PARAMETER_BOUNDS,
            options={
                "maxiter": 100,
                "ftol": 1e-10,
                "gtol": 1e-6,
                "maxls": 30,
            },
        )

        if (
            best_fit is None
            or fit.fun
            < best_fit.fun
        ):
            best_fit = fit

    estimated_parameters = np.exp(
        best_fit.x
    )

    fitted_weekly_mean = (
        deterministic_expected_incidence(
            population_size=
                population_size,
            susceptible_initial=
                susceptible_initial,
            infectious_initial=
                infectious_initial,
            b=float(
                estimated_parameters[0]
            ),
            gamma=float(
                estimated_parameters[1]
            ),
        )
    )

    return {
        "N":
            population_size,
        "S0":
            susceptible_initial,
        "I0":
            infectious_initial,
        "R0":
            R0_FIXED,
        "screening_nll":
            float(
                best_fit.fun
            ),
        "screening_b":
            float(
                estimated_parameters[0]
            ),
        "screening_gamma":
            float(
                estimated_parameters[1]
            ),
        "screening_phi":
            float(
                estimated_parameters[2]
            ),
        "fitted_total_cases":
            float(
                fitted_weekly_mean.sum()
            ),
        "screening_success":
            bool(
                best_fit.success
            ),
        "screening_iterations":
            int(
                best_fit.nit
            ),
    }


DETERMINISTIC_START = time.perf_counter()

deterministic_rows = []

for configuration_number, (
    candidate_population,
    candidate_susceptible,
    candidate_infectious,
) in enumerate(
    ALL_CONFIGURATIONS,
    start=1,
):
    result = (
        fit_deterministic_configuration(
            population_size=
                candidate_population,
            susceptible_initial=
                candidate_susceptible,
            infectious_initial=
                candidate_infectious,
        )
    )

    deterministic_rows.append(
        result
    )

    if (
        configuration_number % 20
        == 0
    ):
        print(
            "Estimated",
            configuration_number,
            "of",
            len(
                ALL_CONFIGURATIONS
            ),
            "deterministic configurations",
        )


DETERMINISTIC_RUNTIME = (
    time.perf_counter()
    - DETERMINISTIC_START
)

DETERMINISTIC_RESULTS = (
    pd.DataFrame(
        deterministic_rows
    )
    .sort_values(
        [
            "screening_nll",
            "N",
            "I0",
        ]
    )
    .reset_index(drop=True)
)

DETERMINISTIC_RESULTS.to_csv(
    OUTPUT_DIR
    / "deterministic_profile_results.csv",
    index=False,
)

print()
print(
    "Best deterministic configurations"
)

display(
    DETERMINISTIC_RESULTS.head(
        20
    ).round(6)
)


# ============================================================
# 8. SELECT BEST DETERMINISTIC CONFIGURATION
# ============================================================

BEST_DETERMINISTIC = (
    DETERMINISTIC_RESULTS.iloc[0]
)

N_SELECTED = int(
    BEST_DETERMINISTIC["N"]
)

S0_SELECTED = int(
    BEST_DETERMINISTIC["S0"]
)

I0_SELECTED = int(
    BEST_DETERMINISTIC["I0"]
)

R0_SELECTED = int(
    BEST_DETERMINISTIC["R0"]
)

assert (
    N_SELECTED
    == S0_SELECTED
    + I0_SELECTED
    + R0_SELECTED
)

DETERMINISTIC_B = float(
    BEST_DETERMINISTIC[
        "screening_b"
    ]
)

DETERMINISTIC_GAMMA = float(
    BEST_DETERMINISTIC[
        "screening_gamma"
    ]
)

DETERMINISTIC_PHI = float(
    BEST_DETERMINISTIC[
        "screening_phi"
    ]
)

DETERMINISTIC_PNLL = float(
    BEST_DETERMINISTIC[
        "screening_nll"
    ]
)

print()
print(
    "Selected deterministic configuration"
)

print(
    {
        "N_eff":
            N_SELECTED,
        "S0":
            S0_SELECTED,
        "I0":
            I0_SELECTED,
        "R0":
            R0_SELECTED,
        "b_deterministic":
            DETERMINISTIC_B,
        "gamma_deterministic":
            DETERMINISTIC_GAMMA,
        "phi_deterministic":
            DETERMINISTIC_PHI,
        "deterministic_penalized_nll":
            DETERMINISTIC_PNLL,
    }
)


# ============================================================
# 9. EXACT FINITE-POPULATION CTMC MODEL
# ============================================================

@dataclass
class ExactRewardModel:
    population_size: int
    susceptible_initial: int
    infectious_initial: int
    infection_matrix: csr_matrix
    recovery_matrix: csr_matrix
    initial_vector: np.ndarray
    infection_trace: float
    recovery_trace: float
    number_of_states: int

    @classmethod
    def create(
        cls,
        population_size: int,
        susceptible_initial: int,
        infectious_initial: int,
    ) -> "ExactRewardModel":
        removed_initial = (
            population_size
            - susceptible_initial
            - infectious_initial
        )

        if removed_initial != R0_FIXED:
            raise ValueError(
                "The exact configuration violates R(0)=60."
            )

        states = []

        for susceptible in range(
            susceptible_initial + 1
        ):
            maximum_infectious = (
                infectious_initial
                + susceptible_initial
                - susceptible
            )

            for infectious in range(
                maximum_infectious + 1
            ):
                states.append(
                    (
                        susceptible,
                        infectious,
                    )
                )

        state_index = {
            state: index
            for index, state
            in enumerate(states)
        }

        number_of_states = len(
            states
        )

        infection_rows = []
        infection_columns = []
        infection_values = []

        recovery_rows = []
        recovery_columns = []
        recovery_values = []

        infection_diagonal = np.zeros(
            number_of_states,
            dtype=float,
        )

        recovery_diagonal = np.zeros(
            number_of_states,
            dtype=float,
        )

        infection_reward = np.zeros(
            number_of_states,
            dtype=float,
        )

        for source_index, (
            susceptible,
            infectious,
        ) in enumerate(states):
            if (
                susceptible > 0
                and infectious > 0
            ):
                destination_index = (
                    state_index[
                        (
                            susceptible - 1,
                            infectious + 1,
                        )
                    ]
                )

                coefficient = (
                    susceptible
                    * infectious
                    / population_size
                )

                infection_rows.append(
                    source_index
                )

                infection_columns.append(
                    destination_index
                )

                infection_values.append(
                    coefficient
                )

                infection_diagonal[
                    source_index
                ] -= coefficient

                infection_reward[
                    source_index
                ] = coefficient

            if infectious > 0:
                destination_index = (
                    state_index[
                        (
                            susceptible,
                            infectious - 1,
                        )
                    ]
                )

                coefficient = float(
                    infectious
                )

                recovery_rows.append(
                    source_index
                )

                recovery_columns.append(
                    destination_index
                )

                recovery_values.append(
                    coefficient
                )

                recovery_diagonal[
                    source_index
                ] -= coefficient

        state_numbers = np.arange(
            number_of_states
        )

        infection_generator = coo_matrix(
            (
                np.concatenate(
                    [
                        np.asarray(
                            infection_values,
                            dtype=float,
                        ),
                        infection_diagonal,
                    ]
                ),
                (
                    np.concatenate(
                        [
                            np.asarray(
                                infection_rows,
                                dtype=int,
                            ),
                            state_numbers,
                        ]
                    ),
                    np.concatenate(
                        [
                            np.asarray(
                                infection_columns,
                                dtype=int,
                            ),
                            state_numbers,
                        ]
                    ),
                ),
            ),
            shape=(
                number_of_states,
                number_of_states,
            ),
        ).tocsr()

        recovery_generator = coo_matrix(
            (
                np.concatenate(
                    [
                        np.asarray(
                            recovery_values,
                            dtype=float,
                        ),
                        recovery_diagonal,
                    ]
                ),
                (
                    np.concatenate(
                        [
                            np.asarray(
                                recovery_rows,
                                dtype=int,
                            ),
                            state_numbers,
                        ]
                    ),
                    np.concatenate(
                        [
                            np.asarray(
                                recovery_columns,
                                dtype=int,
                            ),
                            state_numbers,
                        ]
                    ),
                ),
            ),
            shape=(
                number_of_states,
                number_of_states,
            ),
        ).tocsr()

        zero_column = csr_matrix(
            (
                number_of_states,
                1,
            )
        )

        zero_reward_row = csr_matrix(
            (
                1,
                number_of_states,
            )
        )

        infection_augmented = bmat(
            [
                [
                    infection_generator.T,
                    zero_column,
                ],
                [
                    csr_matrix(
                        infection_reward.reshape(
                            1,
                            -1,
                        )
                    ),
                    csr_matrix(
                        (1, 1)
                    ),
                ],
            ],
            format="csr",
        )

        recovery_augmented = bmat(
            [
                [
                    recovery_generator.T,
                    zero_column,
                ],
                [
                    zero_reward_row,
                    csr_matrix(
                        (1, 1)
                    ),
                ],
            ],
            format="csr",
        )

        initial_vector = np.zeros(
            number_of_states + 1,
            dtype=float,
        )

        initial_vector[
            state_index[
                (
                    susceptible_initial,
                    infectious_initial,
                )
            ]
        ] = 1.0

        return cls(
            population_size=
                population_size,
            susceptible_initial=
                susceptible_initial,
            infectious_initial=
                infectious_initial,
            infection_matrix=
                infection_augmented,
            recovery_matrix=
                recovery_augmented,
            initial_vector=
                initial_vector,
            infection_trace=float(
                infection_augmented
                .diagonal()
                .sum()
            ),
            recovery_trace=float(
                recovery_augmented
                .diagonal()
                .sum()
            ),
            number_of_states=
                number_of_states,
        )

    def expected_incidence(
        self,
        b: float,
        gamma: float,
    ) -> np.ndarray:
        augmented_generator = (
            b
            * self.infection_matrix
            + gamma
            * self.recovery_matrix
        )

        trace_value = (
            b
            * self.infection_trace
            + gamma
            * self.recovery_trace
        )

        equal_intervals = np.allclose(
            INTERVAL_LENGTHS,
            INTERVAL_LENGTHS[0],
            atol=1e-12,
        )

        if equal_intervals:
            interval_length = float(
                INTERVAL_LENGTHS[0]
            )

            trajectory = expm_multiply(
                augmented_generator,
                self.initial_vector,
                start=0.0,
                stop=(
                    interval_length
                    * len(
                        INTERVAL_LENGTHS
                    )
                ),
                num=(
                    len(
                        INTERVAL_LENGTHS
                    )
                    + 1
                ),
                endpoint=True,
                traceA=trace_value,
            )

            cumulative_reward = (
                trajectory[
                    :,
                    -1,
                ]
            )

            return np.maximum(
                np.diff(
                    cumulative_reward
                ),
                1e-12,
            )

        current_vector = (
            self.initial_vector.copy()
        )

        previous_reward = 0.0
        interval_means = []

        for interval_length in (
            INTERVAL_LENGTHS
        ):
            current_vector = expm_multiply(
                augmented_generator
                * float(
                    interval_length
                ),
                current_vector,
                traceA=(
                    trace_value
                    * float(
                        interval_length
                    )
                ),
            )

            current_reward = float(
                current_vector[-1]
            )

            interval_means.append(
                max(
                    current_reward
                    - previous_reward,
                    1e-12,
                )
            )

            previous_reward = (
                current_reward
            )

        return np.asarray(
            interval_means,
            dtype=float,
        )


FINAL_MODEL = ExactRewardModel.create(
    population_size=N_SELECTED,
    susceptible_initial=S0_SELECTED,
    infectious_initial=I0_SELECTED,
)

print()
print(
    "Exact CTMC state-space size",
    FINAL_MODEL.number_of_states,
)


# ============================================================
# 10. EXACT CONDITIONAL LIKELIHOOD
# ============================================================

def create_exact_objective(
    observations: np.ndarray,
) -> Callable[
    [
        np.ndarray
    ],
    float,
]:
    observations = np.asarray(
        observations,
        dtype=float,
    )

    cache = {}

    def objective(
        eta: np.ndarray,
    ) -> float:
        eta = np.asarray(
            eta,
            dtype=float,
        )

        key = tuple(
            np.round(
                eta,
                10,
            )
        )

        if key in cache:
            return cache[key]

        if np.any(
            ~np.isfinite(eta)
        ):
            return 1e100

        b = float(
            np.exp(
                eta[0]
            )
        )

        gamma = float(
            np.exp(
                eta[1]
            )
        )

        try:
            expected_mean = (
                FINAL_MODEL
                .expected_incidence(
                    b=b,
                    gamma=gamma,
                )
            )

            value = (
                penalized_nll_from_mean(
                    eta=eta,
                    expected_mean=
                        expected_mean,
                    observations=
                        observations,
                )
            )

        except Exception:
            value = 1e100

        cache[key] = float(
            value
        )

        return float(
            value
        )

    return objective


EXACT_FIT_START = time.perf_counter()

FINAL_OBJECTIVE = (
    create_exact_objective(
        observations=Y
    )
)

exact_starts = [
    np.array(
        [
            DETERMINISTIC_B,
            DETERMINISTIC_GAMMA,
            DETERMINISTIC_PHI,
        ],
        dtype=float,
    ),
    *EXACT_PARAMETER_STARTS,
]

best_exact_fit = None

for parameter_start in exact_starts:
    exact_fit = minimize(
        FINAL_OBJECTIVE,
        x0=np.log(
            parameter_start
        ),
        method="L-BFGS-B",
        bounds=LOG_PARAMETER_BOUNDS,
        options={
            "maxiter": 180,
            "ftol": 1e-12,
            "gtol": 1e-7,
            "maxls": 40,
        },
    )

    if (
        best_exact_fit is None
        or exact_fit.fun
        < best_exact_fit.fun
    ):
        best_exact_fit = exact_fit


if (
    best_exact_fit is None
    or not np.isfinite(
        best_exact_fit.fun
    )
    or best_exact_fit.fun
    >= 1e90
):
    raise RuntimeError(
        "The exact conditional optimization failed."
    )


ETA_HAT = np.asarray(
    best_exact_fit.x,
    dtype=float,
)

B_HAT = float(
    np.exp(
        ETA_HAT[0]
    )
)

GAMMA_HAT = float(
    np.exp(
        ETA_HAT[1]
    )
)

PHI_HAT = float(
    np.exp(
        ETA_HAT[2]
    )
)

EXACT_PNLL = float(
    best_exact_fit.fun
)

FITTED_WEEKLY_MEAN = (
    FINAL_MODEL
    .expected_incidence(
        b=B_HAT,
        gamma=GAMMA_HAT,
    )
)

EXACT_FIT_RUNTIME = (
    time.perf_counter()
    - EXACT_FIT_START
)


print()
print(
    "Exact conditional estimates"
)

print(
    {
        "N_eff":
            N_SELECTED,
        "S0":
            S0_SELECTED,
        "I0":
            I0_SELECTED,
        "R0":
            R0_SELECTED,
        "b_hat":
            B_HAT,
        "gamma_hat":
            GAMMA_HAT,
        "phi_hat":
            PHI_HAT,
        "exact_penalized_nll":
            EXACT_PNLL,
        "optimizer_success":
            bool(
                best_exact_fit.success
            ),
        "optimizer_message":
            str(
                best_exact_fit.message
            ),
    }
)


# ============================================================
# 11. NUMERICAL GRADIENT AND HESSIAN
# ============================================================

def numerical_gradient(
    function: Callable[
        [
            np.ndarray
        ],
        float,
    ],
    point: np.ndarray,
    relative_step: float = 1e-5,
) -> np.ndarray:
    point = np.asarray(
        point,
        dtype=float,
    )

    steps = (
        relative_step
        * np.maximum(
            1.0,
            np.abs(
                point
            ),
        )
    )

    gradient = np.zeros_like(
        point
    )

    for index in range(
        len(point)
    ):
        direction = np.zeros_like(
            point
        )

        direction[index] = (
            steps[index]
        )

        gradient[index] = (
            function(
                point + direction
            )
            - function(
                point - direction
            )
        ) / (
            2.0
            * steps[index]
        )

    return gradient


def numerical_hessian(
    function: Callable[
        [
            np.ndarray
        ],
        float,
    ],
    point: np.ndarray,
    relative_step: float,
) -> np.ndarray:
    point = np.asarray(
        point,
        dtype=float,
    )

    dimension = len(
        point
    )

    steps = (
        relative_step
        * np.maximum(
            1.0,
            np.abs(
                point
            ),
        )
    )

    hessian = np.zeros(
        (
            dimension,
            dimension,
        ),
        dtype=float,
    )

    central_value = function(
        point
    )

    for first in range(
        dimension
    ):
        first_direction = np.zeros(
            dimension,
            dtype=float,
        )

        first_direction[first] = (
            steps[first]
        )

        hessian[
            first,
            first,
        ] = (
            function(
                point
                + first_direction
            )
            - 2.0
            * central_value
            + function(
                point
                - first_direction
            )
        ) / (
            steps[first] ** 2
        )

    for first in range(
        dimension
    ):
        for second in range(
            first + 1,
            dimension,
        ):
            first_direction = np.zeros(
                dimension,
                dtype=float,
            )

            second_direction = np.zeros(
                dimension,
                dtype=float,
            )

            first_direction[first] = (
                steps[first]
            )

            second_direction[second] = (
                steps[second]
            )

            mixed_value = (
                function(
                    point
                    + first_direction
                    + second_direction
                )
                - function(
                    point
                    + first_direction
                    - second_direction
                )
                - function(
                    point
                    - first_direction
                    + second_direction
                )
                + function(
                    point
                    - first_direction
                    - second_direction
                )
            ) / (
                4.0
                * steps[first]
                * steps[second]
            )

            hessian[
                first,
                second,
            ] = mixed_value

            hessian[
                second,
                first,
            ] = mixed_value

    return (
        0.5
        * (
            hessian
            + hessian.T
        )
    )


HESSIAN_START = time.perf_counter()

GRADIENT_AT_OPTIMUM = numerical_gradient(
    FINAL_OBJECTIVE,
    ETA_HAT,
)

hessian_candidates = []

for hessian_step in [
    1e-3,
    5e-4,
    2.5e-4,
]:
    candidate_hessian = numerical_hessian(
        FINAL_OBJECTIVE,
        ETA_HAT,
        relative_step=
            hessian_step,
    )

    candidate_eigenvalues = (
        np.linalg.eigvalsh(
            candidate_hessian
        )
    )

    hessian_candidates.append(
        {
            "step":
                hessian_step,
            "hessian":
                candidate_hessian,
            "eigenvalues":
                candidate_eigenvalues,
            "positive_definite":
                bool(
                    np.min(
                        candidate_eigenvalues
                    )
                    > 0.0
                ),
            "condition_number":
                (
                    np.linalg.cond(
                        candidate_hessian
                    )
                    if np.min(
                        candidate_eigenvalues
                    ) > 0.0
                    else np.inf
                ),
        }
    )


valid_hessians = [
    candidate
    for candidate
    in hessian_candidates
    if candidate[
        "positive_definite"
    ]
]

if not valid_hessians:
    raise RuntimeError(
        "No positive-definite Hessian was obtained."
    )

SELECTED_HESSIAN_RESULT = min(
    valid_hessians,
    key=lambda candidate:
        candidate[
            "condition_number"
        ],
)

HESSIAN_ETA = (
    SELECTED_HESSIAN_RESULT[
        "hessian"
    ]
)

HESSIAN_EIGENVALUES = (
    SELECTED_HESSIAN_RESULT[
        "eigenvalues"
    ]
)

SIGMA_ETA = np.linalg.inv(
    HESSIAN_ETA
)

SIGMA_BG = SIGMA_ETA[
    np.ix_(
        [
            0,
            1,
        ],
        [
            0,
            1,
        ],
    )
]

HESSIAN_RUNTIME = (
    time.perf_counter()
    - HESSIAN_START
)


print()
print(
    "Gradient at exact optimum"
)

print(
    GRADIENT_AT_OPTIMUM
)

print()
print(
    "Hessian eigenvalues"
)

print(
    HESSIAN_EIGENVALUES
)

print()
print(
    "Covariance matrix of log b, log gamma, log phi"
)

print(
    SIGMA_ETA
)


np.savetxt(
    OUTPUT_DIR
    / "Hessian_log_b_log_gamma_log_phi.csv",
    HESSIAN_ETA,
    delimiter=",",
)

np.savetxt(
    OUTPUT_DIR
    / "Sigma_log_b_log_gamma_log_phi.csv",
    SIGMA_ETA,
    delimiter=",",
)

np.savetxt(
    OUTPUT_DIR
    / "Sigma_log_b_log_gamma.csv",
    SIGMA_BG,
    delimiter=",",
)


# ============================================================
# 12. EXACT FINAL-SIZE DISTRIBUTION
# ============================================================

def final_size_pmf(
    population_size: int,
    susceptible_initial: int,
    infectious_initial: int,
    b: float,
    gamma: float,
) -> np.ndarray:
    current_probability = np.zeros(
        infectious_initial + 1,
        dtype=float,
    )

    current_probability[
        infectious_initial
    ] = 1.0

    pmf = np.zeros(
        susceptible_initial + 1,
        dtype=float,
    )

    for susceptible in range(
        susceptible_initial,
        -1,
        -1,
    ):
        maximum_infectious = (
            infectious_initial
            + susceptible_initial
            - susceptible
        )

        required_length = (
            maximum_infectious
            + 1
        )

        if len(
            current_probability
        ) < required_length:
            current_probability = np.pad(
                current_probability,
                (
                    0,
                    required_length
                    - len(
                        current_probability
                    ),
                ),
            )

        if susceptible > 0:
            infection_component = (
                b
                * susceptible
                / population_size
            )

            infection_probability = (
                infection_component
                / (
                    infection_component
                    + gamma
                )
            )

            next_probability = np.zeros(
                maximum_infectious + 2,
                dtype=float,
            )

        else:
            infection_probability = 0.0
            next_probability = None

        recovery_probability = (
            1.0
            - infection_probability
        )

        for infectious in range(
            maximum_infectious,
            0,
            -1,
        ):
            mass = current_probability[
                infectious
            ]

            if mass == 0.0:
                continue

            current_probability[
                infectious - 1
            ] += (
                mass
                * recovery_probability
            )

            if susceptible > 0:
                next_probability[
                    infectious + 1
                ] += (
                    mass
                    * infection_probability
                )

        number_of_new_infections = (
            susceptible_initial
            - susceptible
        )

        pmf[
            number_of_new_infections
        ] = current_probability[0]

        if susceptible > 0:
            current_probability = (
                next_probability
            )

    pmf = np.maximum(
        pmf,
        0.0,
    )

    pmf = (
        pmf
        / pmf.sum()
    )

    return pmf


def strict_tail_probability(
    pmf: np.ndarray,
) -> np.ndarray:
    non_strict_tail = np.cumsum(
        pmf[::-1]
    )[::-1]

    return (
        non_strict_tail
        - pmf
    )


# ============================================================
# 13. EXACT EXTINCTION-TIME MOMENTS
# ============================================================

def extinction_time_moments(
    population_size: int,
    susceptible_initial: int,
    infectious_initial: int,
    b: float,
    gamma: float,
) -> dict:
    previous_mean = None
    previous_second = None

    for susceptible in range(
        susceptible_initial + 1
    ):
        maximum_infectious = (
            infectious_initial
            + susceptible_initial
            - susceptible
        )

        mean = np.zeros(
            maximum_infectious + 1,
            dtype=float,
        )

        second = np.zeros(
            maximum_infectious + 1,
            dtype=float,
        )

        infection_component = (
            b
            * susceptible
            / population_size
        )

        denominator = (
            infection_component
            + gamma
        )

        infection_probability = (
            infection_component
            / denominator
            if susceptible > 0
            else 0.0
        )

        recovery_probability = (
            1.0
            - infection_probability
        )

        for infectious in range(
            1,
            maximum_infectious + 1,
        ):
            total_rate = (
                infectious
                * denominator
            )

            holding_mean = (
                1.0
                / total_rate
            )

            recovery_mean = mean[
                infectious - 1
            ]

            recovery_second = second[
                infectious - 1
            ]

            if susceptible > 0:
                infection_mean = (
                    previous_mean[
                        infectious + 1
                    ]
                )

                infection_second = (
                    previous_second[
                        infectious + 1
                    ]
                )

            else:
                infection_mean = 0.0
                infection_second = 0.0

            next_mean = (
                infection_probability
                * infection_mean
                + recovery_probability
                * recovery_mean
            )

            next_second = (
                infection_probability
                * infection_second
                + recovery_probability
                * recovery_second
            )

            mean[infectious] = (
                holding_mean
                + next_mean
            )

            second[infectious] = (
                2.0
                * holding_mean**2
                + 2.0
                * holding_mean
                * next_mean
                + next_second
            )

        previous_mean = mean
        previous_second = second

    mean_value = mean[
        infectious_initial
    ]

    second_value = second[
        infectious_initial
    ]

    variance_value = max(
        second_value
        - mean_value**2,
        0.0,
    )

    return {
        "mean":
            mean_value,
        "variance":
            variance_value,
        "sd":
            np.sqrt(
                variance_value
            ),
    }


# ============================================================
# 14. EPIDEMIC DESCRIPTORS
# ============================================================

RISK_START = time.perf_counter()

PMF_HAT = final_size_pmf(
    population_size=N_SELECTED,
    susceptible_initial=S0_SELECTED,
    infectious_initial=I0_SELECTED,
    b=B_HAT,
    gamma=GAMMA_HAT,
)

STRICT_TAIL_HAT = strict_tail_probability(
    PMF_HAT
)

FINAL_SIZE_VALUES = np.arange(
    len(
        PMF_HAT
    )
)

FINAL_SIZE_MEAN = float(
    np.sum(
        FINAL_SIZE_VALUES
        * PMF_HAT
    )
)

FINAL_SIZE_SECOND = float(
    np.sum(
        FINAL_SIZE_VALUES**2
        * PMF_HAT
    )
)

FINAL_SIZE_VARIANCE = max(
    FINAL_SIZE_SECOND
    - FINAL_SIZE_MEAN**2,
    0.0,
)

FINAL_SIZE_SD = np.sqrt(
    FINAL_SIZE_VARIANCE
)

EXTINCTION_RESULTS = extinction_time_moments(
    population_size=N_SELECTED,
    susceptible_initial=S0_SELECTED,
    infectious_initial=I0_SELECTED,
    b=B_HAT,
    gamma=GAMMA_HAT,
)

RISK_RUNTIME = (
    time.perf_counter()
    - RISK_START
)


# Thresholds are fractions of the final selected S(0)

TAIL_THRESHOLDS = np.unique(
    np.clip(
        np.round(
            TAIL_FRACTIONS
            * S0_SELECTED
        ).astype(int),
        0,
        S0_SELECTED - 1,
    )
)

TAIL_ESTIMATES = (
    STRICT_TAIL_HAT[
        TAIL_THRESHOLDS
    ]
)


# ============================================================
# 15. DELTA-METHOD TAIL STANDARD ERRORS
# ============================================================

def selected_tail_function(
    eta: np.ndarray,
) -> np.ndarray:
    b = float(
        np.exp(
            eta[0]
        )
    )

    gamma = float(
        np.exp(
            eta[1]
        )
    )

    pmf = final_size_pmf(
        population_size=N_SELECTED,
        susceptible_initial=S0_SELECTED,
        infectious_initial=I0_SELECTED,
        b=b,
        gamma=gamma,
    )

    strict_tail = (
        strict_tail_probability(
            pmf
        )
    )

    return strict_tail[
        TAIL_THRESHOLDS
    ]


def numerical_jacobian(
    function: Callable[
        [
            np.ndarray
        ],
        np.ndarray,
    ],
    point: np.ndarray,
    relative_step: float = 1e-5,
) -> np.ndarray:
    point = np.asarray(
        point,
        dtype=float,
    )

    central_value = function(
        point
    )

    jacobian = np.zeros(
        (
            len(
                central_value
            ),
            len(
                point
            ),
        ),
        dtype=float,
    )

    steps = (
        relative_step
        * np.maximum(
            1.0,
            np.abs(
                point
            ),
        )
    )

    for parameter_index in range(
        len(
            point
        )
    ):
        direction = np.zeros_like(
            point
        )

        direction[
            parameter_index
        ] = steps[
            parameter_index
        ]

        jacobian[
            :,
            parameter_index,
        ] = (
            function(
                point + direction
            )
            - function(
                point - direction
            )
        ) / (
            2.0
            * steps[
                parameter_index
            ]
        )

    return jacobian


TAIL_JACOBIAN = numerical_jacobian(
    selected_tail_function,
    ETA_HAT,
)

TAIL_DELTA_STANDARD_ERRORS = np.sqrt(
    np.maximum(
        np.einsum(
            "ki,ij,kj->k",
            TAIL_JACOBIAN,
            SIGMA_ETA,
            TAIL_JACOBIAN,
        ),
        0.0,
    )
)


# ============================================================
# 16. PARAMETRIC-BOOTSTRAP MAX-T BANDS
#
# The selected discrete configuration remains fixed.
# b, gamma, and phi are re-estimated in every bootstrap sample.
# ============================================================

BOOTSTRAP_START = time.perf_counter()

bootstrap_rng = np.random.default_rng(
    BASE_SEED
)

NB_PROBABILITY_HAT = (
    PHI_HAT
    / (
        PHI_HAT
        + FITTED_WEEKLY_MEAN
    )
)

bootstrap_eta_values = []
bootstrap_tail_values = []
bootstrap_rows = []


for bootstrap_index in range(
    N_BOOTSTRAP_REPLICATES
):
    bootstrap_observations = (
        bootstrap_rng
        .negative_binomial(
            n=PHI_HAT,
            p=NB_PROBABILITY_HAT,
        )
        .astype(int)
    )

    bootstrap_objective = (
        create_exact_objective(
            observations=
                bootstrap_observations
        )
    )

    bootstrap_fit = minimize(
        bootstrap_objective,
        x0=ETA_HAT,
        method="L-BFGS-B",
        bounds=LOG_PARAMETER_BOUNDS,
        options={
            "maxiter": 70,
            "ftol": 1e-9,
            "gtol": 1e-5,
            "maxls": 25,
        },
    )

    valid_fit = (
        np.isfinite(
            bootstrap_fit.fun
        )
        and bootstrap_fit.fun
        < 1e90
        and np.all(
            np.isfinite(
                bootstrap_fit.x
            )
        )
    )

    if valid_fit:
        bootstrap_eta = np.asarray(
            bootstrap_fit.x,
            dtype=float,
        )

        bootstrap_tail = (
            selected_tail_function(
                bootstrap_eta
            )
        )

        bootstrap_eta_values.append(
            bootstrap_eta
        )

        bootstrap_tail_values.append(
            bootstrap_tail
        )

        bootstrap_rows.append(
            {
                "replication":
                    bootstrap_index + 1,
                "valid":
                    True,
                "optimizer_success":
                    bool(
                        bootstrap_fit.success
                    ),
                "penalized_nll":
                    float(
                        bootstrap_fit.fun
                    ),
                "b":
                    float(
                        np.exp(
                            bootstrap_eta[0]
                        )
                    ),
                "gamma":
                    float(
                        np.exp(
                            bootstrap_eta[1]
                        )
                    ),
                "phi":
                    float(
                        np.exp(
                            bootstrap_eta[2]
                        )
                    ),
                "bootstrap_total_cases":
                    int(
                        bootstrap_observations.sum()
                    ),
            }
        )

    else:
        bootstrap_rows.append(
            {
                "replication":
                    bootstrap_index + 1,
                "valid":
                    False,
                "optimizer_success":
                    False,
                "penalized_nll":
                    np.nan,
                "b":
                    np.nan,
                "gamma":
                    np.nan,
                "phi":
                    np.nan,
                "bootstrap_total_cases":
                    int(
                        bootstrap_observations.sum()
                    ),
            }
        )

    if (
        bootstrap_index + 1
    ) % 10 == 0:
        print(
            "Bootstrap replication",
            bootstrap_index + 1,
            "of",
            N_BOOTSTRAP_REPLICATES,
        )


BOOTSTRAP_RESULTS = pd.DataFrame(
    bootstrap_rows
)

NUMBER_VALID_BOOTSTRAPS = len(
    bootstrap_tail_values
)

BOOTSTRAP_FAILURE_RATE = (
    1.0
    - NUMBER_VALID_BOOTSTRAPS
    / N_BOOTSTRAP_REPLICATES
)

if (
    BOOTSTRAP_FAILURE_RATE
    > MAX_BOOTSTRAP_FAILURE_RATE
):
    raise RuntimeError(
        "Too many bootstrap optimizations failed. "
        f"Failure rate was "
        f"{BOOTSTRAP_FAILURE_RATE:.3f}."
    )

if NUMBER_VALID_BOOTSTRAPS < 30:
    raise RuntimeError(
        "Too few valid bootstrap replications."
    )


BOOTSTRAP_ETA_MATRIX = np.asarray(
    bootstrap_eta_values,
    dtype=float,
)

BOOTSTRAP_TAIL_MATRIX = np.asarray(
    bootstrap_tail_values,
    dtype=float,
)

BOOTSTRAP_STANDARD_ERRORS = (
    BOOTSTRAP_TAIL_MATRIX.std(
        axis=0,
        ddof=1,
    )
)


# Use delta standard errors for studentization.
#
# The bootstrap distribution captures the joint dependence
# across thresholds.

active_components = (
    TAIL_DELTA_STANDARD_ERRORS
    > 1e-10
)

if not np.any(
    active_components
):
    raise RuntimeError(
        "All selected tail components have zero "
        "delta-method variance."
    )


STANDARDIZED_BOOTSTRAP_DEVIATIONS = (
    BOOTSTRAP_TAIL_MATRIX[
        :,
        active_components,
    ]
    - TAIL_ESTIMATES[
        active_components
    ]
) / TAIL_DELTA_STANDARD_ERRORS[
    active_components
]

BOOTSTRAP_MAX_T = np.max(
    np.abs(
        STANDARDIZED_BOOTSTRAP_DEVIATIONS
    ),
    axis=1,
)

MAX_T_CRITICAL = float(
    np.quantile(
        BOOTSTRAP_MAX_T,
        1.0 - ALPHA,
    )
)


# Pointwise normal-delta intervals

POINTWISE_LOWER = np.clip(
    TAIL_ESTIMATES
    - 1.959964
    * TAIL_DELTA_STANDARD_ERRORS,
    0.0,
    1.0,
)

POINTWISE_UPPER = np.clip(
    TAIL_ESTIMATES
    + 1.959964
    * TAIL_DELTA_STANDARD_ERRORS,
    0.0,
    1.0,
)


# Parametric-bootstrap max-t simultaneous bands

SIMULTANEOUS_LOWER = np.clip(
    TAIL_ESTIMATES
    - MAX_T_CRITICAL
    * TAIL_DELTA_STANDARD_ERRORS,
    0.0,
    1.0,
)

SIMULTANEOUS_UPPER = np.clip(
    TAIL_ESTIMATES
    + MAX_T_CRITICAL
    * TAIL_DELTA_STANDARD_ERRORS,
    0.0,
    1.0,
)


ALEATORY_VARIANCE = (
    TAIL_ESTIMATES
    * (
        1.0
        - TAIL_ESTIMATES
    )
)

EPISTEMIC_VARIANCE = (
    TAIL_DELTA_STANDARD_ERRORS**2
)

KAPPA = np.divide(
    EPISTEMIC_VARIANCE,
    ALEATORY_VARIANCE,
    out=np.full_like(
        EPISTEMIC_VARIANCE,
        np.nan,
    ),
    where=(
        ALEATORY_VARIANCE
        > 1e-12
    ),
)


BOOTSTRAP_RUNTIME = (
    time.perf_counter()
    - BOOTSTRAP_START
)


# ============================================================
# 17. OBSERVATION-MODEL FIT
# ============================================================

NB_PROBABILITY = (
    PHI_HAT
    / (
        PHI_HAT
        + FITTED_WEEKLY_MEAN
    )
)

PREDICTIVE_LOWER = nbinom.ppf(
    0.025,
    PHI_HAT,
    NB_PROBABILITY,
)

PREDICTIVE_UPPER = nbinom.ppf(
    0.975,
    PHI_HAT,
    NB_PROBABILITY,
)

OBSERVATION_FIT = pd.DataFrame(
    {
        "week":
            WEEKLY_DATA["week"],
        "week_start":
            WEEKLY_DATA["week_start"],
        "week_end":
            WEEKLY_DATA["week_end"],
        "observed_cases":
            Y,
        "fitted_mean":
            FITTED_WEEKLY_MEAN,
        "predictive_lower":
            PREDICTIVE_LOWER,
        "predictive_upper":
            PREDICTIVE_UPPER,
        "pearson_residual":
            (
                Y
                - FITTED_WEEKLY_MEAN
            )
            / np.sqrt(
                FITTED_WEEKLY_MEAN
                + FITTED_WEEKLY_MEAN**2
                / PHI_HAT
            ),
    }
)


# ============================================================
# 18. RESULTS TABLES
# ============================================================

PARAMETER_RESULTS = pd.DataFrame(
    {
        "quantity": [
            "N_eff_selected",
            "S0_selected",
            "I0_selected",
            "R0_fixed",
            "psi",
            "number_of_regimes",
            "deterministic_b",
            "deterministic_gamma",
            "deterministic_phi",
            "deterministic_penalized_nll",
            "exact_b_hat",
            "exact_gamma_hat",
            "exact_phi_hat",
            "exact_penalized_nll",
        ],
        "estimate": [
            N_SELECTED,
            S0_SELECTED,
            I0_SELECTED,
            R0_SELECTED,
            PSI,
            NUMBER_OF_REGIMES,
            DETERMINISTIC_B,
            DETERMINISTIC_GAMMA,
            DETERMINISTIC_PHI,
            DETERMINISTIC_PNLL,
            B_HAT,
            GAMMA_HAT,
            PHI_HAT,
            EXACT_PNLL,
        ],
    }
)


APPLICATION_DIAGNOSTICS = pd.DataFrame(
    {
        "quantity": [
            "Number of weekly observations",
            "Observed cases",
            "Deterministic fitted cases",
            "Exact fitted expected cases",
            "Exact fitted minus observed cases",
            "Expected final size",
            "Expected extinction time in weeks",
            "Difference from 24 weeks",
        ],
        "value": [
            T,
            OBSERVED_TOTAL,
            float(
                BEST_DETERMINISTIC[
                    "fitted_total_cases"
                ]
            ),
            float(
                FITTED_WEEKLY_MEAN.sum()
            ),
            float(
                FITTED_WEEKLY_MEAN.sum()
                - OBSERVED_TOTAL
            ),
            FINAL_SIZE_MEAN,
            EXTINCTION_RESULTS[
                "mean"
            ],
            EXTINCTION_RESULTS[
                "mean"
            ]
            - 24.0,
        ],
    }
)


DESCRIPTOR_RESULTS = pd.DataFrame(
    {
        "descriptor": [
            "Expected final size",
            "Final-size variance",
            "Final-size standard deviation",
            "Expected extinction time",
            "Extinction-time variance",
            "Extinction-time standard deviation",
        ],
        "estimate": [
            FINAL_SIZE_MEAN,
            FINAL_SIZE_VARIANCE,
            FINAL_SIZE_SD,
            EXTINCTION_RESULTS[
                "mean"
            ],
            EXTINCTION_RESULTS[
                "variance"
            ],
            EXTINCTION_RESULTS[
                "sd"
            ],
        ],
    }
)


TAIL_RESULTS = pd.DataFrame(
    {
        "requested_fraction_of_S0":
            TAIL_FRACTIONS,
        "threshold":
            TAIL_THRESHOLDS,
        "realized_fraction_of_S0":
            TAIL_THRESHOLDS
            / S0_SELECTED,
        "tail_risk":
            TAIL_ESTIMATES,
        "delta_standard_error":
            TAIL_DELTA_STANDARD_ERRORS,
        "bootstrap_standard_error":
            BOOTSTRAP_STANDARD_ERRORS,
        "pointwise_lower":
            POINTWISE_LOWER,
        "pointwise_upper":
            POINTWISE_UPPER,
        "simultaneous_lower":
            SIMULTANEOUS_LOWER,
        "simultaneous_upper":
            SIMULTANEOUS_UPPER,
        "aleatory_variance":
            ALEATORY_VARIANCE,
        "epistemic_variance":
            EPISTEMIC_VARIANCE,
        "kappa":
            KAPPA,
    }
)


print()
print(
    "Parameter estimates"
)

display(
    PARAMETER_RESULTS.round(6)
)

print()
print(
    "Application diagnostics"
)

display(
    APPLICATION_DIAGNOSTICS.round(6)
)

print()
print(
    "Finite-outbreak descriptors"
)

display(
    DESCRIPTOR_RESULTS.round(6)
)

print()
print(
    "Tail-risk inference"
)

print(
    {
        "requested_bootstrap_replications":
            N_BOOTSTRAP_REPLICATES,
        "valid_bootstrap_replications":
            NUMBER_VALID_BOOTSTRAPS,
        "bootstrap_failure_rate":
            BOOTSTRAP_FAILURE_RATE,
        "bootstrap_max_t_critical_value":
            MAX_T_CRITICAL,
    }
)

display(
    TAIL_RESULTS.round(6)
)


# ============================================================
# 19. FIGURES
# ============================================================

fig, axes = plt.subplots(
    1,
    2,
    figsize=(
        13,
        5,
    ),
)


axes[0].fill_between(
    OBSERVATION_FIT["week"],
    OBSERVATION_FIT[
        "predictive_lower"
    ],
    OBSERVATION_FIT[
        "predictive_upper"
    ],
    color="tab:blue",
    alpha=0.20,
    label="Negative-binomial 95% predictive interval",
)

axes[0].plot(
    OBSERVATION_FIT["week"],
    OBSERVATION_FIT[
        "fitted_mean"
    ],
    color="tab:blue",
    linewidth=2,
    label="Exact fitted weekly mean",
)

axes[0].scatter(
    OBSERVATION_FIT["week"],
    OBSERVATION_FIT[
        "observed_cases"
    ],
    color="black",
    s=32,
    zorder=3,
    label="Observed weekly incidence",
)

axes[0].set_xlabel(
    "Week"
)

axes[0].set_ylabel(
    "Weekly incidence"
)

axes[0].grid(
    alpha=0.20,
)

axes[0].legend(
    frameon=False,
    fontsize=8,
)


axes[1].plot(
    FINAL_SIZE_VALUES,
    STRICT_TAIL_HAT,
    color="black",
    linewidth=2,
    label=r"$\Pr(N^I>c)$",
)

axes[1].errorbar(
    TAIL_THRESHOLDS,
    TAIL_ESTIMATES,
    yerr=np.vstack(
        [
            TAIL_ESTIMATES
            - SIMULTANEOUS_LOWER,
            SIMULTANEOUS_UPPER
            - TAIL_ESTIMATES,
        ]
    ),
    fmt="o",
    capsize=5,
    color="tab:blue",
    label="Simultaneous 95% bands",
)

axes[1].errorbar(
    TAIL_THRESHOLDS,
    TAIL_ESTIMATES,
    yerr=np.vstack(
        [
            TAIL_ESTIMATES
            - POINTWISE_LOWER,
            POINTWISE_UPPER
            - TAIL_ESTIMATES,
        ]
    ),
    fmt="none",
    capsize=3,
    color="tab:orange",
    label="Pointwise 95% intervals",
)

axes[1].set_xlabel(
    "Final-size threshold"
)

axes[1].set_ylabel(
    "Exceedance probability"
)

axes[1].set_ylim(
    -0.01,
    1.01,
)

axes[1].grid(
    alpha=0.20,
)

axes[1].legend(
    frameon=False,
    fontsize=8,
)

fig.tight_layout()

fig.savefig(
    OUTPUT_DIR
    / "experiment1_results.png",
    dpi=300,
    bbox_inches="tight",
)

plt.show()


# Deterministic profile by N

BEST_DETERMINISTIC_BY_N = (
    DETERMINISTIC_RESULTS
    .sort_values(
        [
            "screening_nll",
            "N",
        ]
    )
    .groupby(
        "N",
        as_index=False,
    )
    .first()
    .sort_values(
        "N"
    )
)

fig, ax = plt.subplots(
    figsize=(
        8,
        5,
    )
)

ax.plot(
    BEST_DETERMINISTIC_BY_N[
        "N"
    ],
    BEST_DETERMINISTIC_BY_N[
        "screening_nll"
    ],
    marker="o",
    linewidth=2,
)

ax.axvline(
    N_SELECTED,
    color="tab:red",
    linestyle="--",
    label=(
        f"Selected N_eff = "
        f"{N_SELECTED}"
    ),
)

ax.set_xlabel(
    "Effective population candidate"
)

ax.set_ylabel(
    "Best deterministic penalized negative log-likelihood"
)

ax.grid(
    alpha=0.20,
)

ax.legend(
    frameon=False,
)

fig.tight_layout()

fig.savefig(
    OUTPUT_DIR
    / "deterministic_population_profile.png",
    dpi=300,
    bbox_inches="tight",
)

plt.show()


# ============================================================
# 20. SAVE RESULTS
# ============================================================

PARAMETER_RESULTS.to_csv(
    OUTPUT_DIR
    / "estimated_parameters.csv",
    index=False,
)

APPLICATION_DIAGNOSTICS.to_csv(
    OUTPUT_DIR
    / "application_diagnostics.csv",
    index=False,
)

DESCRIPTOR_RESULTS.to_csv(
    OUTPUT_DIR
    / "epidemic_descriptors.csv",
    index=False,
)

TAIL_RESULTS.to_csv(
    OUTPUT_DIR
    / "tail_risk_inference.csv",
    index=False,
)

OBSERVATION_FIT.to_csv(
    OUTPUT_DIR
    / "observation_fit.csv",
    index=False,
)

BOOTSTRAP_RESULTS.to_csv(
    OUTPUT_DIR
    / "parametric_bootstrap_fits.csv",
    index=False,
)

BEST_DETERMINISTIC_BY_N.to_csv(
    OUTPUT_DIR
    / "deterministic_population_profile.csv",
    index=False,
)

np.savetxt(
    OUTPUT_DIR
    / "bootstrap_eta_matrix.csv",
    BOOTSTRAP_ETA_MATRIX,
    delimiter=",",
)

np.savetxt(
    OUTPUT_DIR
    / "bootstrap_tail_matrix.csv",
    BOOTSTRAP_TAIL_MATRIX,
    delimiter=",",
)


if len(
    BOOTSTRAP_ETA_MATRIX
) > 1:
    BOOTSTRAP_SIGMA_ETA = np.cov(
        BOOTSTRAP_ETA_MATRIX,
        rowvar=False,
    )

    np.savetxt(
        OUTPUT_DIR
        / "bootstrap_covariance_log_parameters.csv",
        BOOTSTRAP_SIGMA_ETA,
        delimiter=",",
    )


pd.DataFrame(
    {
        "new_infections":
            FINAL_SIZE_VALUES,
        "probability":
            PMF_HAT,
        "strict_tail_probability":
            STRICT_TAIL_HAT,
    }
).to_csv(
    OUTPUT_DIR
    / "final_size_distribution.csv",
    index=False,
)


# ============================================================
# 21. EXECUTION TIMES
# ============================================================

TOTAL_RUNTIME = (
    time.perf_counter()
    - TOTAL_START
)

RUNTIME_RESULTS = pd.DataFrame(
    {
        "component": [
            "Deterministic discrete profiling",
            "Exact conditional estimation",
            "Numerical Hessian",
            "Exact epidemic descriptors",
            "Parametric-bootstrap max-t inference",
            "Total",
        ],
        "seconds": [
            DETERMINISTIC_RUNTIME,
            EXACT_FIT_RUNTIME,
            HESSIAN_RUNTIME,
            RISK_RUNTIME,
            BOOTSTRAP_RUNTIME,
            TOTAL_RUNTIME,
        ],
    }
)

RUNTIME_RESULTS.to_csv(
    OUTPUT_DIR
    / "execution_times.csv",
    index=False,
)

print()
print(
    "Execution times"
)

display(
    RUNTIME_RESULTS.round(3)
)

print()
print(
    "Experiment 1 completed"
)

print(
    "Results saved in",
    OUTPUT_DIR,
)

Upload Cameroonmpox.csv


Saving Cameroonmpox.csv to Cameroonmpox.csv
Raw columns
['location', 'date', 'iso_code', 'total_cases_old', 'total_deaths', 'new_cases', 'new_deaths', 'new_cases_smoothed', 'new_deaths_smoothed', 'new_cases_per_million', 'total_cases_per_million', 'new_cases_smoothed_per_million', 'new_deaths_per_million', 'total_deaths_per_million', 'new_deaths_smoothed_per_million', 'total_cases']
Date column date
Cumulative-count column total_cases

Weekly data summary
{'number_of_weeks': 24, 'total_cases': 48, 'minimum_weekly_cases': 0, 'maximum_weekly_cases': 8}

Weekly incidence vector
[0, 1, 0, 0, 2, 0, 0, 2, 1, 5, 4, 4, 2, 1, 2, 1, 2, 0, 8, 5, 4, 2, 1, 1]


,week,week_label,week_start,week_end,number_of_days,weekly_cases,cumulative_end
0,1,2025-11-16,2025-11-16,2025-11-16,1,0,0
1,2,2025-11-23,2025-11-17,2025-11-23,7,1,1
2,3,2025-11-30,2025-11-24,2025-11-30,7,0,1
3,4,2025-12-07,2025-12-01,2025-12-07,7,0,1
4,5,2025-12-14,2025-12-08,2025-12-14,7,2,3
5,6,2025-12-21,2025-12-15,2025-12-21,7,0,3
6,7,2025-12-28,2025-12-22,2025-12-28,7,0,3
7,8,2026-01-04,2025-12-29,2026-01-04,7,2,5
8,9,2026-01-11,2026-01-05,2026-01-11,7,1,6
9,10,2026-01-18,2026-01-12,2026-01-18,7,5,11



Number of deterministic configurations 60
Estimated 20 of 60 deterministic configurations
Estimated 40 of 60 deterministic configurations
Estimated 60 of 60 deterministic configurations

Best deterministic configurations


,N,S0,I0,R0,screening_nll,screening_b,screening_gamma,screening_phi,fitted_total_cases,screening_success,screening_iterations
0,175,114,1,60,41.129002,0.775864,0.332120,4.294937,49.749560,True,17
1,150,89,1,60,41.301462,0.841828,0.309240,4.143757,46.533636,True,16
2,200,139,1,60,41.392227,0.723121,0.343867,3.968481,50.490264,True,16
3,225,164,1,60,41.694772,0.682076,0.348976,3.625074,50.450889,True,13
4,250,189,1,60,41.953960,0.650714,0.351179,3.362595,50.244890,True,17
5,225,163,2,60,42.179679,0.595947,0.317471,3.622223,50.110293,True,16
6,250,188,2,60,42.193753,0.570737,0.322030,3.565836,50.169009,True,19
7,200,138,2,60,42.246764,0.628696,0.310010,3.616758,49.709731,True,17
8,175,113,2,60,42.553093,0.671781,0.297628,3.431440,48.341942,True,12
9,250,187,3,60,43.164622,0.524428,0.304201,3.032630,50.906602,True,16



Selected deterministic configuration
{'N_eff': 175, 'S0': 114, 'I0': 1, 'R0': 60, 'b_deterministic': 0.7758638805737327, 'gamma_deterministic': 0.3321201379498925, 'phi_deterministic': 4.294937106368881, 'deterministic_penalized_nll': 41.12900234023654}

Exact CTMC state-space size 6785

Exact conditional estimates
{'N_eff': 175, 'S0': 114, 'I0': 1, 'R0': 60, 'b_hat': 0.7650252948799288, 'gamma_hat': 0.2376850205855924, 'phi_hat': 2.4906081365386212, 'exact_penalized_nll': 45.35123870371151, 'optimizer_success': True, 'optimizer_message': 'CONVERGENCE: RELATIVE REDUCTION OF F <= FACTR*EPSMCH'}

Gradient at exact optimum
[-2.36326514e-05  7.47479589e-07 -2.37285747e-06]

Hessian eigenvalues
[  1.77994753  44.47556924 305.62781702]

Covariance matrix of log b, log gamma, log phi
[[ 0.01078625  0.01032878 -0.01802086]
 [ 0.01032878  0.01760121 -0.03299129]
 [-0.01802086 -0.03299129  0.55918307]]
Bootstrap replication 10 of 199
Bootstrap replication 20 of 199
Bootstrap replication 30 of 1

In [3]:
from pathlib import Path
from google.colab import files
import time

RESULTS_DIR = Path("/content/experiment1_cameroon_risk")

if not RESULTS_DIR.exists():
    raise FileNotFoundError(
        f"Results directory not found at {RESULTS_DIR}"
    )

result_files = sorted(
    path
    for path in RESULTS_DIR.iterdir()
    if path.is_file()
)

print("Files to download")

for path in result_files:
    print(path.name)

for path in result_files:
    files.download(str(path))
    time.sleep(0.5)

FileNotFoundError: Results directory not found at /content/experiment1_cameroon_risk